Students implement Hill-Climbing Search, Local Beam Search, and Simulated Annealing Search algorithms following TODO 1 - 3. \
Students can add supporting attributes and methods to the three classes as needed.

# Libraries

In [2]:
import os
import heapq
import math
import random

# Graph class

In [3]:
# Directed, weighted graphs
class Graph:
  def __init__(self):
    self.AL = dict() # adjacency list
    self.V = 0
    self.E = 0
    self.H = dict()

  def __str__(self):
    res = 'V: %d, E: %d\n'%(self.V, self.E)
    for u, neighbors in self.AL.items():
      line = '%d: %s\n'%(u, str(neighbors))
      res += line
    for u, h in self.H.items():
      line = 'h(%d) = %d\n'%(u, h)
      res += line
    return res

  def print(self):
    print(str(self))

  def load_from_file(self, filename):
    '''
        Example input file:
            V E
            u v w
            u v w
            u v w
            ...
            u1 h1
            u2 h2
            u3 h3
            ...

        # input.txt
        7 8
        0 1 5
        0 2 6
        1 3 12
        1 4 9
        2 5 5
        3 5 8
        3 6 7
        4 6 4
        0 14
        1 13
        2 12
        3 11
        4 10
        5 9
        6 0
    '''
    if os.path.exists(filename):
      with open(filename) as g:
        self.V, self.E = [int(it) for it in g.readline().split()]
        for i in range(self.E):
          line = g.readline()
          u, v, w = [int(it) for it in line.strip().split()]
          if u not in self.AL:
            self.AL[u] = []
          self.AL[u].append((v, w))
        for i in range(self.V):
          line = g.readline()
          u, h = [int(it) for it in line.strip().split()]
          self.H[u] = h
  def get_neighbors(self, u):
    return self.AL.get(u, [])

In [4]:
g = Graph()
g.load_from_file('input.txt')
g.print()

V: 7, E: 8
0: [(1, 5), (2, 6)]
1: [(3, 12), (4, 9)]
2: [(5, 5)]
3: [(5, 8), (6, 7)]
4: [(6, 4)]
h(0) = 14
h(1) = 13
h(2) = 12
h(3) = 11
h(4) = 10
h(5) = 9
h(6) = 0



# Search Strategies

In [5]:
class LocalSearchStrategy:
  def search(self, g: Graph, src: int) -> tuple:
    '''
    return a tuple (u, p) in which
      u: the local maximum state
      p: the priority/weight/desirability/cost of u
    '''
    return (None, None)

In [6]:
class HillClimbingSearch(LocalSearchStrategy):
    def search(self, g: Graph, src: int) -> tuple:
        '''
        return a tuple (u, p) in which
          u: the local maximum state
          p: the priority/weight/desirability/cost of u

        Note: weight of a node u = path_cost to u + heuristic value of u (similar to A*)
        '''
        # TODO 1
        current      = src
        path_cost    = 0
        current_p    = path_cost + g.H[current]

        while True:
            neighbors = g.get_neighbors(current)
            if not neighbors:
                break

            # Find the best neighbor (first that strictly improves current priority)
            best = None
            for neighbor, step_cost in neighbors:
                neighbor_p = (path_cost + step_cost) + g.H[neighbor]
                if neighbor_p > current_p:
                    best = (neighbor, step_cost, neighbor_p)
                    break

            # No improving neighbor found → local maximum reached
            if best is None:
                break

            neighbor, step_cost, current_p = best
            path_cost += step_cost
            current    = neighbor

        return (current, current_p)

In [7]:
class LocalBeamSearch(LocalSearchStrategy):
    def __init__(self, n=5):
        self.n = n  # Beam width

    def search(self, g: Graph, src: int) -> tuple:
        '''
        return a tuple (u, p) in which:
          u: the local maximum state
          p: the priority/weight/desirability/cost of u

        Note:
        - weight of a node u = path_cost to u + heuristic value of u (similar to A*)
        - parameter n is provided in the constructor
        '''
        # TODO 2
        # Each entry: (priority, path_cost, node)
        beam = [(g.H[src], 0, src)]

        while True:
            candidates = []

            # Expand every state currently in the beam
            for priority, path_cost, u in beam:
                for neighbor, step_cost in g.get_neighbors(u):
                    new_path_cost = path_cost + step_cost
                    new_priority  = new_path_cost + g.H[neighbor]
                    heapq.heappush(candidates, (new_priority, new_path_cost, neighbor))

            # No further expansions possible → return the best state in beam
            if not candidates:
                best_priority, _, best_node = max(beam, key=lambda x: x[0])
                return (best_node, best_priority)

            # Keep only the top-n candidates for the next iteration
            beam = heapq.nsmallest(self.n, candidates)

In [8]:
class SimulatedAnnealingSearch(LocalSearchStrategy):
    def schedule(self, t):
        return 1 / math.pow(t, 2)  # Cooling schedule: T = 1 / t²

    def search(self, g: Graph, src: int) -> tuple:
        '''
        return a tuple (u, p) in which:
          u: the final state found
          p: the priority/weight/desirability/cost of u

        Notes:
        - Unlike Hill Climbing, SAS allows probabilistic downhill moves.
        - Cooling schedule: schedule(t) = 1 / (t^2)
        '''
        # TODO 3
        current   = src
        path_cost = 0
        current_p = path_cost + g.H[current]

        t = 1  # Iteration step (controls temperature via schedule)

        while True:
            T = self.schedule(t)
            if T < 1e-5:  # Temperature too low → freeze
                break

            neighbors = g.get_neighbors(current)
            if not neighbors:
                break

            # Pick a random neighbor and evaluate it
            neighbor, step_cost  = random.choice(neighbors)
            new_path_cost        = path_cost + step_cost
            new_p                = new_path_cost + g.H[neighbor]

            delta_E = new_p - current_p

            # Accept if better; otherwise accept with probability e^(ΔE / T)
            if delta_E > 0 or random.uniform(0, 1) < math.exp(delta_E / T):
                current   = neighbor
                path_cost = new_path_cost
                current_p = new_p

            t += 1

        return (current, current_p)

# Evaluation

In [9]:
hcs = HillClimbingSearch()
lbs = LocalBeamSearch()
sas = SimulatedAnnealingSearch()

for stg in [hcs, lbs, sas]:
  print(stg)
  u, p = stg.search(g, 0)
  print(u, p)

5 34
5 34
5 20


# Submission

*   Students download the notebook after completion
*   Rename the notebook in which inserting your student ID at the beginning. \
For example, **123456-lec06-LocalSearch-HW.ipynb**
*   Finally, submit the file